In [2]:
import pandas as pd
import os 
path = 'C:/Users/bruno/OneDrive/Documentos/Base de datos/PROYECTO BBDD/'
os.chdir(path)
data_raw = pd.read_csv('Datos/fuente/20250926_Directorio_Oficial_EE_2025_20250430_WEB.csv', encoding='utf-8', sep = ';')
data_raw.head()

,AGNO,RBD,DGV_RBD,NOM_RBD,MRUN,RUT_SOSTENEDOR,P_JURIDICA,COD_REG_RBD,NOM_REG_RBD_A,COD_PRO_RBD,...,ESPE_02,ESPE_03,ESPE_04,ESPE_05,ESPE_06,ESPE_07,ESPE_08,ESPE_09,ESPE_10,ESPE_11
0,2025,1,9,LICEO POLITECNICO ARICA,,62000660,1,15,AYP,151,...,52009,52010,52013,53014,53015,61002,61003,62004,64001,0
1,2025,2,7,PARVULARIO LAS ESPIGUITAS,,62000660,1,15,AYP,151,...,0,0,0,0,0,0,0,0,0,0
2,2025,3,5,ESC. PEDRO VICENTE GUTIERREZ TORRES,,62000660,1,15,AYP,151,...,0,0,0,0,0,0,0,0,0,0
3,2025,4,3,LICEO OCTAVIO PALMA PEREZ,,62000660,1,15,AYP,151,...,0,0,0,0,0,0,0,0,0,0
4,2025,5,1,JOVINA NARANJO FERNANDEZ,,62000660,1,15,AYP,151,...,0,0,0,0,0,0,0,0,0,0


# Creación de tablas del esquema

In [3]:
data = data_raw

data = data.replace(r'^\s*$', None, regex=True)

In [4]:
## Tablas asociadas con ubicación (Region, Provincia, Comuna)
print('-----------------------------------------------------------------------------------------------------------')
### Tabla Region:
region = (
    data[['COD_REG_RBD','NOM_REG_RBD_A']]
    .drop_duplicates('COD_REG_RBD')
    .dropna(subset=['COD_REG_RBD'])
    .reset_index(drop = True)
)
print('Columnas: ',region.columns)
print('Tabla Región: ', region.shape)
print('-----------------------------------------------------------------------------------------------------------')
### Tabla Provincia:
provincia = (
    data[['COD_PRO_RBD','COD_REG_RBD']]
    .drop_duplicates('COD_PRO_RBD')
    .dropna(subset=['COD_PRO_RBD','COD_REG_RBD'])
    .rename(columns={'COD_REG_RBD':'r_COD_REG_RBD'})
    .reset_index(drop=True)
)
print('Columnas: ',provincia.columns)
print('Tabla Provincia: ', provincia.shape)
print('-----------------------------------------------------------------------------------------------------------')
### Tabla Comuna:
comuna = (
    data[['COD_COM_RBD', 'NOM_COM_RBD', 'COD_PRO_RBD']]
    .drop_duplicates('COD_COM_RBD')
    .dropna(subset=['COD_COM_RBD'])
    .rename(columns={'COD_PRO_RBD':'p_COD_PRO_RBD'})
    .reset_index(drop = True)
)
print('Columnas: ',comuna.columns)
print('Tabla Comuna: ', comuna.shape)
print('-----------------------------------------------------------------------------------------------------------')
### tabla sostenedor
data['ID_SOSTENEDOR'] = data['RUT_SOSTENEDOR'].fillna(data['MRUN'])

sostenedor = (
    data.loc[data['ID_SOSTENEDOR'].notna(),['ID_SOSTENEDOR', 'P_JURIDICA']]
    .drop_duplicates(subset=['ID_SOSTENEDOR'])
    .rename(columns={'ID_SOSTENEDOR':'RUT'})
    .reset_index(drop=True)
)
print('Columnas: ',sostenedor.columns)
print('Tabla sostenedor: ', sostenedor.shape)
print('-----------------------------------------------------------------------------------------------------------')
### Tabla departamento provincial
deprov = (
    data[['COD_DEPROV_RBD','COD_PRO_RBD','NOM_DEPROV_RBD']]
    .drop_duplicates()
    .dropna(subset=['COD_DEPROV_RBD'])
    .rename(columns={'COD_PRO_RBD':'p_COD_PRO_RBD'})
    .drop_duplicates(subset=['COD_DEPROV_RBD'])
    .reset_index(drop = True)
)
print('Columnas: ',deprov.columns)
print('Tabla Departamento provincial: ', deprov.shape)
print('-----------------------------------------------------------------------------------------------------------')
especialidades = {
    0: 'Ciclo General / Sin información',
    41001: 'Administración',
    41002: 'Contabilidad',
    41003: 'Secretariado',
    41004: 'Ventas',
    41005: 'Administración (con mención)',
    51001: 'Edificación',
    51002: 'Terminaciones de Construcción',
    51003: 'Montaje Industrial',
    51004: 'Obras viales y de infraestructura',
    51005: 'Instalaciones Sanitarias',
    51006: 'Refrigeración y climatización',
    51009: 'Construcción (con mención)',
    52008: 'Mecánica Industrial',
    52009: 'Construcciones Metálicas',
    52010: 'Mecánica Automotriz',
    52011: 'Matricería',
    52012: 'Mecánica de mantención de aeronaves',
    52013: 'Mecánica Industrial (con mención)',
    53014: 'Electricidad',
    53015: 'Electrónica',
    54018: 'Explotación minera',
    54019: 'Metalurgia Extractiva',
    54020: 'Asistencia en geología',
    55022: 'Gráfica',
    55023: 'Dibujo Técnico',
    56025: 'Operación de planta química',
    56026: 'Laboratorio químico',
    56027: 'Química Industrial (con mención)',
    57028: 'Tejido',
    57029: 'Textil',
    57030: 'Vestuario y Confección Textil',
    57031: 'Productos del cuero',
    58033: 'Conectividad y Redes',
    58034: 'Programación',
    58035: 'Telecomunicaciones',
    61001: 'Elaboración Industrial de Alimentos',
    61002: 'Servicio de Alimentación Colectiva',
    61003: 'Gastronomía (con mención)',
    62004: 'Atención de párvulos',
    62005: 'Atención de adultos mayores',
    62006: 'Atención de Enfermería',
    62007: 'Atención Social y Recreativa',
    62008: 'Atención de Enfermería (con mención)',
    63009: 'Servicio de turismo',
    63010: 'Servicios Hoteleros',
    63011: 'Servicio de hotelería',
    71001: 'Forestal',
    71002: 'Procesamiento de la madera',
    71003: 'Productos de la madera',
    71004: 'Celulosa y Papel',
    71005: 'Muebles y Terminaciones de la Madera',
    72006: 'Agropecuaria',
    72007: 'Agropecuaria (con mención)',
    81001: 'Naves mercantes y especiales',
    81002: 'Pesquería',
    81003: 'Acuicultura',
    81004: 'Operación portuaria',
    81005: 'Tripulación naves mercantes y especiales',
    91001: 'Artes Visuales',
    91002: 'Artes Audiovisuales',
    91003: 'Diseño',
    92004: 'Interpretación Teatral',
    92005: 'Diseño Escénico',
    93006: 'Interpretación en Danza de Nivel Intermedio',
    93007: 'Monitoría de Danza'
}
# Columnas de especialidad existentes en la base
cols_especialidad = [col for col in data.columns if col.startswith("ESPE_")]

# Crear tabla de especialidades solo con los códigos observados
especialidad = (
    data[cols_especialidad]
    .melt(value_name="COD_ESPE")          # Pasar de formato ancho a largo
    [["COD_ESPE"]]                        # Dejar solo la columna de códigos
    .dropna()                             # Eliminar nulos
)

# Convertir a entero
especialidad["COD_ESPE"] = especialidad["COD_ESPE"].astype(int)

especialidad = (
    especialidad
    .drop_duplicates()
    .reset_index(drop=True)
)
especialidad["NOMBRE_ESPECIALIDAD"] = (
    especialidad["COD_ESPE"]
    .map(especialidades)
)

print('Columnas: ', especialidad.columns)
print('Tabla Especialidad: ', especialidad.shape)
print('-----------------------------------------------------------------------------------------------------------')
ensenanza = pd.DataFrame({
    'COD_ENSE': [
        0, 10, 110, 160, 161, 163, 165, 167,
        211, 212, 213, 214, 215, 216, 217, 218, 219, 299,
        310, 360, 361, 362, 363,410, 460, 461, 463,
        510, 560, 561, 563,610, 660, 661, 663,
        710, 760, 761, 763,810, 860, 863,910, 963],
    'DESCRIPCION': [
        'No Aplica','Educación Parvularia','Enseñanza Básica',
        'Educación Básica Común Adultos','Educación Básica Especial Adultos',
        'Escuelas Cárceles Básica Adultos','Educación Básica Adultos Sin Oficios',
        'Educación Básica Adultos Con Oficios','Educación Especial Discapacidad Auditiva',
        'Educación Especial Discapacidad Intelectual','Educación Especial Discapacidad Visual',
        'Educación Especial Trastornos Específicos del Lenguaje','Educación Especial Trastornos Motores',
        'Educación Especial Autismo','Educación Especial Graves Alteraciones en Relación y Comunicación',
        'Educación Especial Discapacidad Múltiple','Educación Especial Sordoceguera',
        'Opción 4 Programa Integración Escolar','Enseñanza Media HC niños y jóvenes',
        'Educación Media HC adultos vespertino y nocturno','Educación Media HC adultos',
        'Escuelas Cárceles Media Adultos','Educación Media HC Adultos',
        'Enseñanza Media TP Comercial niños y jóvenes','Educación Media TP Comercial Adultos',
        'Educación Media TP Comercial Adultos','Educación Media TP Comercial Adultos',
        'Enseñanza Media TP Industrial niños y jóvenes','Educación Media TP Industrial Adultos',
        'Educación Media TP Industrial Adultos','Educación Media TP Industrial Adultos',
        'Enseñanza Media TP Técnica niños y jóvenes','Educación Media TP Técnica Adultos',
        'Educación Media TP Técnica Adultos','Educación Media TP Técnica Adultos',
        'Enseñanza Media TP Agrícola niños y jóvenes','Educación Media TP Agrícola Adultos',
        'Educación Media TP Agrícola Adultos','Educación Media TP Agrícola Adultos',
        'Enseñanza Media TP Marítima niños y jóvenes','Educación Media TP Marítima Adultos',
        'Educación Media TP Marítima Adultos','Enseñanza Media Artística niños y jóvenes',
        'Enseñanza Media Artística Adultos']
})
print('Columnas: ', especialidad.columns)
print('Tabla Especialidad: ', especialidad.shape)
print('-----------------------------------------------------------------------------------------------------------')
nivel_educativo = pd.DataFrame(
    {'ID_NIVEL': [1, 2, 3, 4, 5, 6, 7, 8],
     'NOMBRE_NIVEL': ['Parvularia',
                      'Básica Regular',
                      'Básica Adulto',
                      'Especial',
                      'Media HC Jóvenes',
                      'Media HC Adultos',
                      'Media TP Jóvenes',
                      'Media TP Adultos'
                    ]
     }
)
print('Columnas: ', nivel_educativo.columns)
print('Tabla nivel_educativo: ', nivel_educativo.shape)
print('-----------------------------------------------------------------------------------------------------------')
establecimiento_ensenanza = (
    data[['RBD'] + [col for col in data.columns if col.startswith("ENS_")]]
    .melt(
        id_vars = 'RBD',
        value_vars = [col for col in data.columns if col.startswith("ENS_")],
        var_name = 'COLUMNA_ORIGEN',
        value_name = 'COD_ENSE'
    )
    .dropna(subset=['COD_ENSE'])
    .sort_values('COD_ENSE')
    .drop_duplicates()
    .reset_index(drop = True)
    .rename(columns={'RBD':'e_RBD','COD_ENSE': 'en_COD_ENSE'})
)
establecimiento_ensenanza = establecimiento_ensenanza[establecimiento_ensenanza['en_COD_ENSE'] != 0]

establecimiento_ensenanza = establecimiento_ensenanza[['e_RBD','en_COD_ENSE']]

print('Columnas: ', establecimiento_ensenanza.columns)
print('Tabla establecimiento_enseñanza: ', establecimiento_ensenanza.shape)
print('-----------------------------------------------------------------------------------------------------------')
establecimiento_especialidad = (
    data[['RBD'] + [col for col in data.columns if col.startswith("ESPE_")]]
    .melt(
        id_vars = 'RBD',
        value_vars = [col for col in data.columns if col.startswith("ESPE_")],
        var_name = 'COLUMNA_ORIGEN',
        value_name = 'COD_ESPE'
    )
    .dropna(subset=['COD_ESPE'])
    .sort_values('COD_ESPE')
    .drop_duplicates()
    .reset_index(drop = True)
    .rename(columns={'RBD':'e_RBD','COD_ESPE': 'es_COD_ESPE'})
)
establecimiento_especialidad = establecimiento_especialidad[establecimiento_especialidad['es_COD_ESPE'] != 0]

establecimiento_especialidad = establecimiento_especialidad[['e_RBD','es_COD_ESPE']]

print('Columnas: ', establecimiento_especialidad.columns)
print('Tabla establecimiento_especialidad: ', establecimiento_especialidad.shape)
print('-----------------------------------------------------------------------------------------------------------')
cols_matricula_nivel = {
    "MAT_ENS_1": 1,
    "MAT_ENS_2": 2,
    "MAT_ENS_3": 3,
    "MAT_ENS_4": 4,
    "MAT_ENS_5": 5,
    "MAT_ENS_6": 6,
    "MAT_ENS_7": 7,
    "MAT_ENS_8": 8
}

establecimiento_nivel = (
    data[['RBD'] + [col for col in data.columns if col.startswith("MAT_ENS")]]
    .melt(
        id_vars = 'RBD',
        value_vars = [col for col in data.columns if col.startswith("MAT_ENS")],
        var_name = 'COLUMNA_ORIGEN',
        value_name = 'CANTIDAD_MATRICULA'
    )
)
establecimiento_nivel["ID_NIVEL"] = establecimiento_nivel["COLUMNA_ORIGEN"].map(cols_matricula_nivel)
establecimiento_nivel = establecimiento_nivel.dropna(subset=["CANTIDAD_MATRICULA"])
establecimiento_nivel = establecimiento_nivel[establecimiento_nivel["CANTIDAD_MATRICULA"] > 0]
establecimiento_nivel = (
    establecimiento_nivel[["RBD", "ID_NIVEL", "CANTIDAD_MATRICULA"]]
    .drop_duplicates()
    .reset_index(drop=True)
    .rename(columns={'RBD': 'e_RBD','ID_NIVEL' : 'n_ID_NIVEL', 'CANTIDAD_MATRICULA': 'CANTIDAD_NIVEL'})
)

print('Columnas: ', establecimiento_nivel.columns)
print('Tabla establecimiento_nivel: ', establecimiento_nivel.shape)
print('-----------------------------------------------------------------------------------------------------------')
establecimiento_sostenedor = (
    data[['RBD', 'ID_SOSTENEDOR']]
    .drop_duplicates()
    .dropna(subset=['RBD'])
    .rename(columns={'RBD':'e_RBD', 'ID_SOSTENEDOR': 's_RUT'})
    .reset_index(drop = True)
)
print('Columnas: ', establecimiento_sostenedor.columns)
print('Tabla establecimiento_sostenedor: ', establecimiento_sostenedor.shape)
print('-----------------------------------------------------------------------------------------------------------')

data["LATITUD"] = pd.to_numeric(
    data["LATITUD"]
        .astype(str)
        .str.replace(",", ".", regex=False),
    errors="coerce"
)

data["LONGITUD"] = pd.to_numeric(
    data["LONGITUD"]
        .astype(str)
        .str.replace(",", ".", regex=False),
    errors="coerce"
)
establecimiento = (
    data[['RBD','COD_COM_RBD','DGV_RBD', 'NOM_RBD',
       'COD_DEPE', 'COD_DEPE2', 'RURAL_RBD', 'LATITUD', 'LONGITUD',
       'CONVENIO_PIE', 'PACE', 'MAT_TOTAL', 'MATRICULA',
       'ESTADO_ESTAB', 'ORI_RELIGIOSA', 'ORI_OTRO_GLOSA', 'PAGO_MATRICULA',
       'PAGO_MENSUAL']]
    .rename(columns={'COD_COM_RBD':'c_COD_COM_RBD'})
    .drop_duplicates(subset=['RBD'])
    .dropna(subset=['RBD'])
    .reset_index(drop = True)
)
print('Columnas: ', establecimiento.columns)
print('Tabla establecimiento: ', establecimiento.shape)

-----------------------------------------------------------------------------------------------------------
Columnas:  Index(['COD_REG_RBD', 'NOM_REG_RBD_A'], dtype='str')
Tabla Región:  (16, 2)
-----------------------------------------------------------------------------------------------------------
Columnas:  Index(['COD_PRO_RBD', 'r_COD_REG_RBD'], dtype='str')
Tabla Provincia:  (56, 2)
-----------------------------------------------------------------------------------------------------------
Columnas:  Index(['COD_COM_RBD', 'NOM_COM_RBD', 'p_COD_PRO_RBD'], dtype='str')
Tabla Comuna:  (346, 3)
-----------------------------------------------------------------------------------------------------------
Columnas:  Index(['RUT', 'P_JURIDICA'], dtype='str')
Tabla sostenedor:  (7884, 2)
-----------------------------------------------------------------------------------------------------------
Columnas:  Index(['COD_DEPROV_RBD', 'p_COD_PRO_RBD', 'NOM_DEPROV_RBD'], dtype='str')
Tabla Departa

In [5]:
## Tablas exportadas

region.to_csv('Datos/Tabla_region.csv',index=False,encoding='latin-1',sep = ';')
provincia.to_csv('Datos/Tabla_provincia.csv',index=False,encoding='latin-1',sep = ';')
comuna.to_csv('Datos/Tabla_comuna.csv',index=False,encoding='latin-1',sep = ';')
sostenedor.to_csv('Datos/Tabla_sostenedor.csv',index=False,encoding='latin-1',sep = ';')
deprov.to_csv('Datos/Tabla_deprov.csv',index=False,encoding='latin-1',sep = ';')
nivel_educativo.to_csv('Datos/Tabla_nivel_educativo.csv',index=False,encoding='latin-1',sep = ';')
especialidad.to_csv('Datos/Tabla_especialidad.csv',index=False,encoding='latin-1',sep = ';')
ensenanza.to_csv('Datos/Tabla_ensenanza.csv',index=False,encoding='latin-1',sep = ';')
establecimiento_ensenanza.to_csv('Datos/Tabla_establecimiento_ensenanza.csv',index=False,encoding='latin-1',sep = ';')
establecimiento_especialidad.to_csv('Datos/Tabla_establecimiento_especialidad.csv',index=False,encoding='latin-1',sep = ';')
establecimiento_nivel.to_csv('Datos/Tabla_establecimiento_nivel.csv',index=False,encoding='latin-1',sep = ';')
establecimiento_sostenedor.to_csv('Datos/Tabla_establecimiento_sostenedor.csv',index=False,encoding='latin-1',sep = ';')
establecimiento.to_csv('Datos/Tabla_establecimiento.csv',index=False,encoding='latin-1',sep = ';')

In [6]:
establecimiento[
    ~establecimiento["LATITUD"].astype(str).str.match(r"^-?\d+([.,]\d+)?$")
].sort_values('LATITUD')

,RBD,c_COD_COM_RBD,DGV_RBD,NOM_RBD,COD_DEPE,COD_DEPE2,RURAL_RBD,LATITUD,LONGITUD,CONVENIO_PIE,PACE,MAT_TOTAL,MATRICULA,ESTADO_ESTAB,ORI_RELIGIOSA,ORI_OTRO_GLOSA,PAGO_MATRICULA,PAGO_MENSUAL
37,44,15101,2,JARDIN INFANTIL PEQUENILANDIA,3,2,0,NaN,NaN,0,0,0,0,3,9,NaN,SIN INFORMACION,SIN INFORMACION
39,46,15101,9,JARDIN INFANTIL SAN ANDRES,3,2,0,NaN,NaN,0,0,0,0,3,2,NaN,GRATUITO,GRATUITO
47,57,15101,4,JARDIN INFANTIL ZORZALITO,3,2,0,NaN,NaN,0,0,0,0,3,9,NaN,SIN INFORMACION,SIN INFORMACION
51,65,15101,5,J. I. SOLDADITO DE PLOMO,4,3,0,NaN,NaN,0,0,0,0,2,9,NaN,SIN INFORMACION,SIN INFORMACION
78,98,1101,1,CENTRO DE DIAGNOSTICO,3,2,0,NaN,NaN,0,0,0,0,3,9,NaN,SIN INFORMACION,SIN INFORMACION
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16762,42391,13403,2,ESCUELA BASICA COLEGIO PAKRITI,4,3,0,NaN,NaN,0,0,0,0,1,9,NaN,SIN INFORMACION,SIN INFORMACION
16763,42392,4101,0,SALA CUNA Y JARDIN INFANTIL COLIBRI,4,3,0,NaN,NaN,0,0,0,0,1,9,NaN,SIN INFORMACION,SIN INFORMACION
16764,42393,10303,9,SALA CUNA MANITOS DE COLORES,4,3,0,NaN,NaN,0,0,0,0,1,9,NaN,SIN INFORMACION,SIN INFORMACION
16766,42396,13101,3,JARDIN INFANTIL MARIA VICTORIA PERALTA,4,3,0,NaN,NaN,0,0,0,0,1,9,NaN,SIN INFORMACION,SIN INFORMACION


## pwd admin1234

In [6]:
data.columns

Index(['AGNO', 'RBD', 'DGV_RBD', 'NOM_RBD', 'MRUN', 'RUT_SOSTENEDOR',
       'P_JURIDICA', 'COD_REG_RBD', 'NOM_REG_RBD_A', 'COD_PRO_RBD',
       'COD_COM_RBD', 'NOM_COM_RBD', 'COD_DEPROV_RBD', 'NOM_DEPROV_RBD',
       'COD_DEPE', 'COD_DEPE2', 'RURAL_RBD', 'LATITUD', 'LONGITUD',
       'CONVENIO_PIE', 'PACE', 'ENS_01', 'ENS_02', 'ENS_03', 'ENS_04',
       'ENS_05', 'ENS_06', 'ENS_07', 'ENS_08', 'ENS_09', 'ENS_10', 'ENS_11',
       'MAT_ENS_1', 'MAT_ENS_2', 'MAT_ENS_3', 'MAT_ENS_4', 'MAT_ENS_5',
       'MAT_ENS_6', 'MAT_ENS_7', 'MAT_ENS_8', 'MAT_TOTAL', 'MATRICULA',
       'ESTADO_ESTAB', 'ORI_RELIGIOSA', 'ORI_OTRO_GLOSA', 'PAGO_MATRICULA',
       'PAGO_MENSUAL', 'ESPE_01', 'ESPE_02', 'ESPE_03', 'ESPE_04', 'ESPE_05',
       'ESPE_06', 'ESPE_07', 'ESPE_08', 'ESPE_09', 'ESPE_10', 'ESPE_11',
       'ID_SOSTENEDOR'],
      dtype='str')